In [ ]:
import os
import sys
from IPython import get_ipython
import pandas as pd
import importlib

# Add parent directory to path to import from helpers
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))

# Only add if not already in path
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

import helpers.process_data
from helpers.process_data import process_data, preprocess_data, in_fold_process_data

# Import NLP helpers
import helpers.nlp
from helpers.nlp import TextFeatureExtractor, combine_text_columns

# Import NLP caching utilities for efficiency
import helpers.nlp_cache
from helpers.nlp_cache import get_or_create_text_features, clear_all_caches

import catboost as cb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')


In [3]:
# Data preparation - no preprocessing yet (done in CV to avoid leakage)
# Extract target from training data
y_train = train['target']

print("Data loaded:")
print(f"Training data shape: {train.shape}")
print(f"Test data shape: {test.shape}")
print("Target distribution:")
print(y_train.value_counts().sort_index())

print("\nTarget distribution percentages:")
target_counts = y_train.value_counts().sort_index()
for label, count in target_counts.items():
    percentage = (count / len(y_train)) * 100
    print(f"  Class {label}: {count:,} samples ({percentage:.1f}%)")

Data loaded:
Training data shape: (26206, 19)
Test data shape: (11232, 19)
Target distribution:
target
0    17423
1     8783
Name: count, dtype: int64

Target distribution percentages:
  Class 0: 17,423 samples (66.5%)
  Class 1: 8,783 samples (33.5%)


In [4]:
# CatBoost with 5-fold Cross Validation - WITH NLP FEATURES
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

# Initialize CatBoost classifier parameters
base_catboost_params = {
    'random_state': 42,
    'iterations': 100,
    'depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bylevel': 0.8,
    'verbose': False,
    'eval_metric': 'Logloss'
}

# Text feature extraction parameters (tunable)
text_params = {
    'max_features_word': 30000,
    'max_features_char': 30000,
    'n_components': 300
}

print("=" * 70)
print("5-FOLD CROSS VALIDATION WITH STRUCTURED + TEXT FEATURES")
print("=" * 70)
print(f"\nText feature parameters:")
print(f"  Word TF-IDF max_features: {text_params['max_features_word']}")
print(f"  Char TF-IDF max_features: {text_params['max_features_char']}")
print(f"  SVD n_components: {text_params['n_components']}")

# Prepare raw data
raw_train_features = train.drop('target', axis=1)
y_train = train['target']

# Prepare text data (combine columns once)
text_columns = ['description', 'screen_name', 'location']
train_text = combine_text_columns(train, columns=text_columns)

print(f"\nRaw training data shape: {raw_train_features.shape}")
print(f"Target shape: {y_train.shape}")
print(f"Combined text shape: {train_text.shape}")

# STEP 1: Pre-CV preprocessing (no data leakage - done once before CV)
print("\n" + "="*70)
print("STEP 1: PRE-CV PREPROCESSING (No Leakage)")
print("="*70)
print("Extracting features that don't cause data leakage...")
print("(Textual features, ratios, engineered features)")

X_preprocessed = preprocess_data(raw_train_features, verbose=True, include_statuses_count=True)

print(f"\nPreprocessed data shape: {X_preprocessed.shape}")
print(f"Missing values in preprocessed data: {X_preprocessed.isnull().sum().sum()}")

# STEP 2: Cross-validation with in-fold processing + text features
print("\n" + "="*70)
print("STEP 2: CROSS-VALIDATION WITH IN-FOLD PROCESSING + TEXT FEATURES")
print("="*70)
print("Imputation, encoding, and text feature extraction will be done within each fold")

# 5-fold stratified cross validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores_acc = []
cv_scores_auc = []
fold_details = []

print(f"\nStarting 5-fold cross validation...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_preprocessed, y_train), 1):
    print(f"\n{'='*20} FOLD {fold}/5 {'='*20}")
    
    # Split preprocessed data into train/validation for this fold
    X_train_fold = X_preprocessed.iloc[train_idx]
    X_val_fold = X_preprocessed.iloc[val_idx]
    y_train_fold = y_train.iloc[train_idx]
    y_val_fold = y_train.iloc[val_idx]
    
    # Split text data
    text_train_fold = train_text.iloc[train_idx]
    text_val_fold = train_text.iloc[val_idx]
    
    print(f"Fold {fold} - Train: {X_train_fold.shape}, Val: {X_val_fold.shape}")
    
    # ========== STRUCTURED FEATURES ==========
    # In-fold processing: imputation and encoding (fit on train, transform both)
    print("Processing structured features (imputation & encoding)...")
    X_train_processed, X_val_processed, impute_vals = in_fold_process_data(
        X_train_fold, X_val_fold,
        imputation_strategy='median',
        categorical_encoding='onehot',
        lang_frequency_threshold=0,
        verbose=False
    )
    
    print(f"  Structured - Train: {X_train_processed.shape}, Val: {X_val_processed.shape}")
    
    # ========== TEXT FEATURES ==========
    print("Extracting text features...")
    # Create fresh text extractor for this fold
    text_extractor_fold = TextFeatureExtractor(
        max_features_word=text_params['max_features_word'],
        max_features_char=text_params['max_features_char'],
        n_components=text_params['n_components'],
        verbose=False
    )
    
    # Fit on train fold, transform both train and val
    X_train_fold_text = text_extractor_fold.fit_transform(text_train_fold)
    X_val_fold_text = text_extractor_fold.transform(text_val_fold)
    
    print(f"  Text - Train: {X_train_fold_text.shape}, Val: {X_val_fold_text.shape}")
    
    # ========== COMBINE FEATURES ==========
    print("Combining structured + text features...")
    X_train_fold_combined = np.hstack([X_train_processed.values, X_train_fold_text])
    X_val_fold_combined = np.hstack([X_val_processed.values, X_val_fold_text])
    
    print(f"  Combined - Train: {X_train_fold_combined.shape}, Val: {X_val_fold_combined.shape}")
    
    # Verify no missing values
    if np.isnan(X_train_fold_combined).any() or np.isnan(X_val_fold_combined).any():
        print(f"WARNING: Missing values detected in combined features!")
    
    # Train CatBoost on this fold
    print("Training CatBoost...")
    catboost_fold = cb.CatBoostClassifier(**base_catboost_params)
    catboost_fold.fit(
        X_train_fold_combined, y_train_fold,
        eval_set=(X_val_fold_combined, y_val_fold),
        use_best_model=True
    )
    
    # Validate on this fold
    val_predictions = catboost_fold.predict(X_val_fold_combined)
    val_probabilities = catboost_fold.predict_proba(X_val_fold_combined)[:, 1]
    
    fold_score_acc = accuracy_score(y_val_fold, val_predictions)
    fold_score_auc = roc_auc_score(y_val_fold, val_probabilities)
    
    cv_scores_acc.append(fold_score_acc)
    cv_scores_auc.append(fold_score_auc)
    
    print(f"Fold {fold} - Accuracy: {fold_score_acc:.4f}, AUC: {fold_score_auc:.4f}")
    
    # Store fold details
    fold_details.append({
        'fold': fold,
        'train_size': len(y_train_fold),
        'val_size': len(y_val_fold),
        'accuracy': fold_score_acc,
        'auc': fold_score_auc,
        'features_structured': X_train_processed.shape[1],
        'features_text': X_train_fold_text.shape[1],
        'features_total': X_train_fold_combined.shape[1]
    })

print(f"\n{'='*70}")
print("CROSS VALIDATION RESULTS")
print(f"{'='*70}")

cv_scores_acc = np.array(cv_scores_acc)
cv_scores_auc = np.array(cv_scores_auc)

print("\nAccuracy Scores:")
print(f"  Individual fold scores: {cv_scores_acc}")
print(f"  Mean CV accuracy: {cv_scores_acc.mean():.4f} (+/- {cv_scores_acc.std() * 2:.4f})")
print(f"  Range: {cv_scores_acc.min():.4f} - {cv_scores_acc.max():.4f}")

print("\nAUC Scores:")
print(f"  Individual fold scores: {cv_scores_auc}")
print(f"  Mean CV AUC: {cv_scores_auc.mean():.4f} (+/- {cv_scores_auc.std() * 2:.4f})")
print(f"  Range: {cv_scores_auc.min():.4f} - {cv_scores_auc.max():.4f}")

# Display fold details
print(f"\nFold Details:")
for detail in fold_details:
    print(f"  Fold {detail['fold']}: {detail['train_size']:,} train, {detail['val_size']:,} val")
    print(f"    Structured: {detail['features_structured']}, Text: {detail['features_text']}, "
          f"Total: {detail['features_total']}, Acc: {detail['accuracy']:.4f}, AUC: {detail['auc']:.4f}")

# STEP 3: Train final model on full dataset
print(f"\n{'='*70}")
print("STEP 3: TRAINING FINAL MODEL ON FULL DATASET")
print(f"{'='*70}")

# CRITICAL FIX: Process training and test data together to ensure consistency
# This is the key to avoiding feature mismatch issues

# Pre-process full training data
print("Pre-processing full training dataset...")
X_train_preprocessed_full = preprocess_data(raw_train_features, verbose=False, include_statuses_count=True)

# Pre-process test data
print("Pre-processing test dataset...")
X_test_preprocessed_full = preprocess_data(test, verbose=False, include_statuses_count=True)

# Process BOTH train and test together using in_fold_process_data
# This ensures they get the exact same one-hot encoded columns
print("Processing train + test together (imputation & encoding)...")
X_train_final_struct, X_test_final_struct, impute_vals_final = in_fold_process_data(
    X_train_preprocessed_full, 
    X_test_preprocessed_full,
    imputation_strategy='median',
    categorical_encoding='onehot',
    lang_frequency_threshold=0,
    verbose=True
)

print(f"  Structured train features: {X_train_final_struct.shape}")
print(f"  Structured test features: {X_test_final_struct.shape}")

# Extract text features on full training data
print("\nExtracting text features on full training dataset...")
text_extractor_final = TextFeatureExtractor(
    max_features_word=text_params['max_features_word'],
    max_features_char=text_params['max_features_char'],
    n_components=text_params['n_components'],
    verbose=True
)

X_train_final_text = text_extractor_final.fit_transform(train_text)
print(f"  Text features: {X_train_final_text.shape}")

# Combine features
print("Combining structured + text features...")
X_train_final = np.hstack([X_train_final_struct.values, X_train_final_text])
print(f"  Combined features: {X_train_final.shape}")

print("\nTraining final CatBoost model...")
catboost_model = cb.CatBoostClassifier(**base_catboost_params)
catboost_model.fit(X_train_final, y_train)

# Training metrics
train_predictions = catboost_model.predict(X_train_final)
train_probabilities = catboost_model.predict_proba(X_train_final)[:, 1]
train_accuracy = accuracy_score(y_train, train_predictions)
train_auc = roc_auc_score(y_train, train_probabilities)

print(f"Final model training accuracy: {train_accuracy:.4f}")
print(f"Final model training AUC: {train_auc:.4f}")

# Feature importance
feature_importance = catboost_model.feature_importances_
# Note: We can't easily get feature names for combined array, so just show count
print(f"\nTotal features: {X_train_final.shape[1]}")
print(f"  Structured: {X_train_final_struct.shape[1]}")
print(f"  Text: {X_train_final_text.shape[1]}")

# Store variables for later use
X_train = X_train_final  # For compatibility with evaluation cell
X_test_struct = X_test_final_struct  # Already processed test data (structured part)
imputation_values_final = impute_vals_final  # For test data processing
text_extractor = text_extractor_final  # For test text feature processing

print("\n✓ Model training complete with structured + text features!")
print(f"✓ Test data structured features ready: {X_test_struct.shape}")


5-FOLD CROSS VALIDATION WITH STRUCTURED + TEXT FEATURES

Text feature parameters:
  Word TF-IDF max_features: 30000
  Char TF-IDF max_features: 30000
  SVD n_components: 300

Raw training data shape: (26206, 18)
Target shape: (26206,)
Combined text shape: (26206,)

STEP 1: PRE-CV PREPROCESSING (No Leakage)
Extracting features that don't cause data leakage...
(Textual features, ratios, engineered features)
=== Pre-CV Data Preprocessing (No Leakage) ===
Original dataframe shape: (26206, 18)

Extracting numerical features...
  - favourites_count: extracted (missing: 0)
  - followers_count: extracted (missing: 0)
  - friends_count: extracted (missing: 0)
  - average_tweets_per_day: extracted (missing: 0)
  - account_age_days: extracted (missing: 0)
  - statuses_count: extracted (missing: 0)

Extracting binary features...
  - default_profile: extracted (missing: 0)
  - default_profile_image: extracted (missing: 0)
  - geo_enabled: extracted (missing: 0)
  - verified: extracted (missing: 0)


In [5]:
# Generate predictions on test set with consistent preprocessing + text features
import joblib
import os

print("=" * 60)
print("GENERATING TEST PREDICTIONS (STRUCTURED + TEXT)")
print("=" * 60)

# Test data was already processed during training to ensure consistency!
# We just need to extract text features and combine

print("\nUsing pre-processed test data from training step...")
print(f"  Structured test features: {X_test_struct.shape}")

# Extract text features from test data
print("\nExtracting text features from test data...")
test_text = combine_text_columns(test, columns=['description', 'screen_name', 'location'])
X_test_text = text_extractor.transform(test_text)
print(f"  Text test features: {X_test_text.shape}")

# Combine features
print("\nCombining structured + text features...")
X_test_combined = np.hstack([X_test_struct.values, X_test_text])
print(f"  Combined test features: {X_test_combined.shape}")

print(f"\nTraining data shape: {X_train.shape}")
print(f"Test data shape: {X_test_combined.shape}")

# Ensure feature consistency
print("\nFeature consistency check:")
print(f"  Same number of columns: {X_train.shape[1] == X_test_combined.shape[1]}")

if X_train.shape[1] == X_test_combined.shape[1]:
    print("  ✓ Perfect feature consistency between train and test")
else:
    print("  ⚠ Feature count mismatch - this may cause issues")
    print(f"    Train: {X_train.shape[1]}, Test: {X_test_combined.shape[1]}")

# Generate predictions
print("\nGenerating predictions on test set...")
test_probabilities = catboost_model.predict_proba(X_test_combined)[:, 1]

# Create submission dataframe
submission = pd.DataFrame({
    'index': test['index'],
    'target': test_probabilities
})

# Save submission file to submissions folder
os.makedirs('../submissions', exist_ok=True)
submission_path = '../submissions/CatBoost_NLP.csv'
submission.to_csv(submission_path, index=False)

# Save trained model to models/trained folder
os.makedirs('trained', exist_ok=True)
model_path = 'trained/catboost_model_nlp.joblib'
joblib.dump(catboost_model, model_path)

# Save text extractor for reproducibility
text_extractor_path = 'trained/text_extractor.joblib'
joblib.dump(text_extractor, text_extractor_path)

print(f"\nSubmission file saved to: {submission_path}")
print(f"Trained model saved to: {model_path}")
print(f"Text extractor saved to: {text_extractor_path}")
print("\nPrediction statistics:")
print(f"  Range: {test_probabilities.min():.4f} to {test_probabilities.max():.4f}")
print(f"  Mean:  {test_probabilities.mean():.4f}")
print(f"  Std:   {test_probabilities.std():.4f}")
print(f"  Shape: {submission.shape}")

# Display first few predictions
print("\nFirst 10 predictions:")
print(submission.head(10))


GENERATING TEST PREDICTIONS (STRUCTURED + TEXT)

Step 1: Pre-processing test data (textual features, ratios, etc.)...
Test data preprocessed shape: (11232, 33)
Missing values: 0

Step 2: Applying imputation & encoding (using training statistics)...
  Structured test features: (11232, 33)

Step 3: Extracting text features from test data...
Transforming 11232 texts...
   Output shape: (11232, 300)
  Text test features: (11232, 300)

Step 4: Combining structured + text features...
  Combined test features: (11232, 333)

Training data shape: (26206, 381)
Test data shape: (11232, 333)

Feature consistency check:
  Same number of columns: False
  ⚠ Feature count mismatch - this may cause issues
    Train: 381, Test: 333

Generating predictions on test set...


CatBoostError: catboost/libs/data/model_dataset_compatibility.cpp:72: Feature 336 is present in model but not in pool.

In [ ]:
# Comprehensive Model Evaluation
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

print("=" * 60)
print("COMPREHENSIVE MODEL EVALUATION")
print("=" * 60)

# Generate predictions and probabilities
train_pred_proba = catboost_model.predict_proba(X_train)[:, 1]
train_predictions = catboost_model.predict(X_train)

# 1. Classification Report
print("\n1. CLASSIFICATION REPORT (Training Data):")
print("-" * 40)
print(classification_report(y_train, train_predictions))

# 2. Confusion Matrix
print("2. CONFUSION MATRIX (Training Data):")
print("-" * 40)
cm = confusion_matrix(y_train, train_predictions)
print(f"Confusion Matrix:\n{cm}")

# Extract values for better interpretation
tn, fp, fn, tp = cm.ravel()
print("\nConfusion Matrix Breakdown:")
print(f"True Negatives (TN):  {tn:,}")
print(f"False Positives (FP): {fp:,}")
print(f"False Negatives (FN): {fn:,}")
print(f"True Positives (TP):  {tp:,}")

# 3. Additional Metrics
print("\n3. DETAILED METRICS:")
print("-" * 40)
from sklearn.metrics import precision_score, recall_score, f1_score

accuracy = accuracy_score(y_train, train_predictions)
precision = precision_score(y_train, train_predictions)
recall = recall_score(y_train, train_predictions)
f1 = f1_score(y_train, train_predictions)
roc_auc = roc_auc_score(y_train, train_pred_proba)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC AUC:   {roc_auc:.4f}")

# 4. Calculate Average Precision (PR AUC)
avg_precision = average_precision_score(y_train, train_pred_proba)
print(f"PR AUC:    {avg_precision:.4f}")

# 5. Class Distribution Analysis
print("\n4. CLASS DISTRIBUTION ANALYSIS:")
print("-" * 40)
class_counts = y_train.value_counts().sort_index()
print("True class distribution:")
for class_label, count in class_counts.items():
    percentage = (count / len(y_train)) * 100
    print(f"  Class {class_label}: {count:,} samples ({percentage:.1f}%)")

print("\nPredicted class distribution:")
pred_counts = pd.Series(train_predictions).value_counts().sort_index()
for class_label, count in pred_counts.items():
    percentage = (count / len(train_predictions)) * 100
    print(f"  Class {class_label}: {count:,} samples ({percentage:.1f}%)")

# 6. Probability Distribution Analysis
print("\n5. PROBABILITY DISTRIBUTION ANALYSIS:")
print("-" * 40)
print("Probability statistics:")
print(f"  Min probability:  {train_pred_proba.min():.4f}")
print(f"  Max probability:  {train_pred_proba.max():.4f}")
print(f"  Mean probability: {train_pred_proba.mean():.4f}")
print(f"  Std probability:  {train_pred_proba.std():.4f}")

# Analyze predictions by class
bot_probs = train_pred_proba[y_train == 1]
human_probs = train_pred_proba[y_train == 0]

print("\nProbability by true class:")
print(f"  Bots (Class 1) - Mean: {bot_probs.mean():.4f}, Std: {bot_probs.std():.4f}")
print(f"  Humans (Class 0) - Mean: {human_probs.mean():.4f}, Std: {human_probs.std():.4f}")

# 7. Model Performance Summary
print("\n6. MODEL PERFORMANCE SUMMARY:")
print("-" * 40)
print("Cross-Validation Performance:")
print(f"  Mean CV Accuracy: {cv_scores_acc.mean():.4f} (+/- {cv_scores_acc.std() * 2:.4f})")
print(f"  CV Accuracy Range: {cv_scores_acc.min():.4f} - {cv_scores_acc.max():.4f}")
print(f"  Mean CV AUC: {cv_scores_auc.mean():.4f} (+/- {cv_scores_auc.std() * 2:.4f})")
print(f"  CV AUC Range: {cv_scores_auc.min():.4f} - {cv_scores_auc.max():.4f}")

print("\nTraining Performance:")
print(f"  Training Accuracy: {accuracy:.4f}")
print(f"  Training ROC AUC:  {roc_auc:.4f}")

# Check for potential overfitting
cv_mean_acc = cv_scores_acc.mean()
overfitting_diff = accuracy - cv_mean_acc
print("\nOverfitting Check:")
print(f"  CV vs Train Accuracy Difference: {overfitting_diff:.4f}")
if overfitting_diff > 0.05:
    print("  Potential overfitting detected (difference > 0.05)")
elif overfitting_diff > 0.02:
    print("  Mild overfitting detected (difference > 0.02)")
else:
    print("  No significant overfitting detected")

print("\n" + "=" * 60)
print("EVALUATION COMPLETE")
print("=" * 60)


In [ ]:
# Plot feature importances (uses importance_df created earlier)
# Show top N features as a horizontal bar chart
top_n = 50
top_df = importance_df.head(top_n).iloc[::-1]  # reverse for plotting

plt.figure(figsize=(8, 0.4 * len(top_df) + 2))
plt.barh(top_df['feature'], top_df['importance'], color='C0')
plt.xlabel('Importance')
plt.title(f'Top {top_n} Feature Importances')
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# Hyperparameter Optimization

### Optuna new should be faster

In [ ]:
# Optuna imports and GPU check
import optuna
from optuna.samplers import TPESampler
import time

# Check GPU availability for CatBoost
try:
    import catboost as cb_check
    # Test if GPU is available by training a small dummy model
    dummy_train = cb_check.Pool([[1, 2], [3, 4]], label=[0, 1])
    test_params = {'task_type': 'GPU', 'devices': '0', 'iterations': 1, 'verbose': False}
    try:
        cb_check.CatBoostClassifier(**test_params).fit(dummy_train)
        gpu_available = True
        print("✓ GPU available for CatBoost")
    except Exception as e:
        gpu_available = False
        print(f"✗ GPU not available for CatBoost: {e}")
        print("Will use CPU instead")
except Exception as e:
    gpu_available = False
    print(f"✗ Could not check GPU availability: {e}")

In [ ]:
# Define Optuna optimization function for CatBoost + NLP
def perform_optuna_catboost_nlp_optimization(raw_train_features, y_train, n_trials=50, cv_folds=5, use_gpu=True):
    """
    Perform hyperparameter optimization for CatBoost using Optuna with:
    - Structured features (from process_data)
    - Text features (from TextFeatureExtractor) WITH CACHING
    - Fold-wise preprocessing to prevent data leakage
    
    Parameters:
    -----------
    raw_train_features : pd.DataFrame
        Raw training features (before preprocessing)
    y_train : pd.Series
        Training target
    n_trials : int, default=50
        Number of optimization trials
    cv_folds : int, default=5
        Number of cross-validation folds
    use_gpu : bool, default=True
        Whether to use GPU acceleration if available
    
    Returns:
    --------
    tuple : (best_model, best_params, best_score, study, X_train_full, feature_list_full, 
             importance_df_optuna, text_extractor_final, best_text_params)
    """
    
    print("="*70)
    print("OPTUNA HYPERPARAMETER OPTIMIZATION (CATBOOST + NLP)")
    print("="*70)
    print("\nOptimizing:")
    print("  - CatBoost hyperparameters (iterations, depth, learning_rate, etc.)")
    print("  - Text feature parameters (max_features_word, max_features_char, n_components)")
    print("\nPreprocessing happens within each CV fold to prevent data leakage.")
    print("Text features cached after final model training for speed!")
    
    # Determine task type based on GPU availability
    if use_gpu and gpu_available:
        task_type = 'GPU'
        print(f"\nUsing GPU acceleration with task_type='{task_type}'")
    else:
        task_type = 'CPU'
        print(f"\nUsing CPU with task_type='{task_type}'")
    
    # Prepare text data once (doesn't need to be in fold loop)
    text_columns = ['description', 'screen_name', 'location']
    train_text = combine_text_columns(raw_train_features, columns=text_columns)
    
    # Setup cross-validation
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    
    print(f"\nOptimization Overview:")
    print(f"   Number of trials: {n_trials}")
    print(f"   Cross-validation folds: {cv_folds}")
    print(f"   Total model fits: {n_trials * cv_folds}")
    print(f"   Estimated time: {(n_trials * 40) / 60:.1f}-{(n_trials * 100) / 60:.1f} minutes\n")
    
    # Define objective function for Optuna
    def objective(trial):
        # Suggest CatBoost hyperparameters
        cb_params = {
            'iterations': trial.suggest_int('iterations', 100, 500, step=50),
            'depth': trial.suggest_int('depth', 4, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
            'random_strength': trial.suggest_float('random_strength', 0.0, 10.0),
            'border_count': trial.suggest_int('border_count', 32, 255),
            'random_state': 42,
            'verbose': False,
            'task_type': task_type
        }
        
        if use_gpu and gpu_available:
            cb_params['devices'] = '0'
        
        # Suggest text feature extraction hyperparameters
        text_params = {
            'max_features_word': trial.suggest_categorical('max_features_word', [10000, 20000, 30000, 50000]),
            'max_features_char': trial.suggest_categorical('max_features_char', [10000, 20000, 30000, 50000]),
            'n_components': trial.suggest_categorical('n_components', [100, 200, 300, 400, 500])
        }
        
        # Perform cross-validation with fold-wise preprocessing
        fold_scores = []
        master_feature_list = None
        
        for fold, (train_idx, val_idx) in enumerate(skf.split(raw_train_features, y_train), 1):
            # Split raw data
            raw_train_fold = raw_train_features.iloc[train_idx]
            raw_val_fold = raw_train_features.iloc[val_idx]
            y_train_fold = y_train.iloc[train_idx]
            y_val_fold = y_train.iloc[val_idx]
            
            # Split text data
            text_train_fold = train_text.iloc[train_idx]
            text_val_fold = train_text.iloc[val_idx]
            
            # ========== STRUCTURED FEATURES ==========
            X_train_fold_struct, fold_feature_list = process_data(raw_train_fold, verbose=False)
            
            # Establish or verify master feature list
            if master_feature_list is None:
                master_feature_list = fold_feature_list.copy()
            else:
                if set(fold_feature_list) != set(master_feature_list):
                    X_train_fold_struct = process_data(
                        raw_train_fold, verbose=False, expected_features=master_feature_list
                    )
            
            X_val_fold_struct = process_data(
                raw_val_fold, verbose=False, expected_features=master_feature_list
            )
            
            # ========== TEXT FEATURES (NO CACHING in CV - prevents leakage) ==========
            text_extractor_fold = TextFeatureExtractor(
                max_features_word=text_params['max_features_word'],
                max_features_char=text_params['max_features_char'],
                n_components=text_params['n_components'],
                verbose=False
            )
            
            X_train_fold_text = text_extractor_fold.fit_transform(text_train_fold)
            X_val_fold_text = text_extractor_fold.transform(text_val_fold)
            
            # ========== COMBINE FEATURES ==========
            X_train_fold = np.hstack([X_train_fold_struct.values, X_train_fold_text])
            X_val_fold = np.hstack([X_val_fold_struct.values, X_val_fold_text])
            
            # Train model
            model = cb.CatBoostClassifier(**cb_params)
            model.fit(X_train_fold, y_train_fold)
            
            # Validate
            val_probabilities = model.predict_proba(X_val_fold)[:, 1]
            fold_score = roc_auc_score(y_val_fold, val_probabilities)
            fold_scores.append(fold_score)
        
        # Return mean CV score
        mean_score = np.mean(fold_scores)
        return mean_score
    
    # Create Optuna study
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=42)
    )
    
    # Run optimization
    print("Starting optimization...")
    start_time = time.time()
    
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    total_time = time.time() - start_time
    
    # Get best parameters
    best_params = study.best_params
    best_score = study.best_value
    
    # Separate CatBoost and text params
    best_text_params = {
        'max_features_word': best_params['max_features_word'],
        'max_features_char': best_params['max_features_char'],
        'n_components': best_params['n_components']
    }
    
    best_cb_params = {k: v for k, v in best_params.items() 
                      if k not in ['max_features_word', 'max_features_char', 'n_components']}
    
    print("\n" + "="*70)
    print("OPTIMIZATION COMPLETE!")
    print("="*70)
    print(f"\nTotal time: {total_time/60:.1f} minutes ({total_time:.0f} seconds)")
    print(f"Best Cross-Validation AUC: {best_score:.4f}")
    
    print("\nBest CatBoost Parameters:")
    for param, value in best_cb_params.items():
        print(f"   {param}: {value}")
    
    print("\nBest Text Feature Parameters:")
    for param, value in best_text_params.items():
        print(f"   {param}: {value}")
    
    # Train final model with best parameters on full dataset
    print("\n" + "="*70)
    print("TRAINING FINAL MODEL WITH BEST PARAMETERS")
    print("="*70)
    
    best_cb_params_full = best_cb_params.copy()
    best_cb_params_full.update({
        'random_state': 42,
        'verbose': False,
        'task_type': task_type
    })
    if use_gpu and gpu_available:
        best_cb_params_full['devices'] = '0'
    
    # Process full training data - structured features
    print("\nProcessing structured features on full training dataset...")
    X_train_full_struct, feature_list_full = process_data(raw_train_features, verbose=False)
    print(f"  Structured features: {X_train_full_struct.shape}")
    
    # Process full training data - text features WITH CACHING
    print("Extracting text features on full training dataset (with caching)...")
    print("Note: Caching is safe here - fitting on FULL training set after CV")
    
    text_extractor_final, X_train_full_text, _ = get_or_create_text_features(
        raw_train_features, test, text_columns, best_text_params, use_cache=True
    )
    
    print(f"  Text features: {X_train_full_text.shape}")
    
    # Combine features
    print("Combining structured + text features...")
    X_train_full = np.hstack([X_train_full_struct.values, X_train_full_text])
    print(f"  Combined features: {X_train_full.shape}")
    
    print("\nTraining final model...")
    best_model = cb.CatBoostClassifier(**best_cb_params_full)
    best_model.fit(X_train_full, y_train)
    
    # Training metrics
    train_probabilities = best_model.predict_proba(X_train_full)[:, 1]
    train_auc = roc_auc_score(y_train, train_probabilities)
    train_predictions = (train_probabilities > 0.5).astype(int)
    train_accuracy = accuracy_score(y_train, train_predictions)
    print(f"Final model training AUC: {train_auc:.4f}")
    print(f"Final model training accuracy: {train_accuracy:.4f}")
    
    # Feature importance
    feature_importance = best_model.feature_importances_
    n_struct = X_train_full_struct.shape[1]
    n_text = X_train_full_text.shape[1]
    
    struct_names = list(X_train_full_struct.columns)
    text_names = [f'text_svd_{i}' for i in range(n_text)]
    all_feature_names = struct_names + text_names
    
    importance_df_optuna = pd.DataFrame({
        'feature': all_feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    print("\nTop 15 Most Important Features:")
    print(importance_df_optuna.head(15).to_string(index=False, float_format='%.4f'))
    
    # Optimization history analysis
    print("\n" + "="*70)
    print("OPTIMIZATION HISTORY ANALYSIS")
    print("="*70)
    
    trials_df = study.trials_dataframe()
    print(f"\nNumber of completed trials: {len(study.trials)}")
    print(f"AUC range: {trials_df['value'].min():.4f} - {trials_df['value'].max():.4f}")
    print(f"AUC std: {trials_df['value'].std():.4f}")
    print(f"Best trial number: {study.best_trial.number + 1}")
    
    # Show top 5 trials
    top_trials = trials_df.nlargest(5, 'value')[['value', 'number'] + 
                                                  [col for col in trials_df.columns if col.startswith('params_')]]
    print("\nTop 5 Trials:")
    print(top_trials.to_string(index=False, float_format='%.4f'))
    
    return (best_model, best_params, best_score, study, X_train_full, 
            feature_list_full, importance_df_optuna, text_extractor_final, best_text_params)

print("✓ Optuna optimization function defined with caching support")

In [ ]:
# Run Optuna optimization
optuna_results = perform_optuna_catboost_nlp_optimization(
    raw_train_features=raw_train_features,
    y_train=y_train,
    n_trials=50,
    cv_folds=5,
    use_gpu=gpu_available
)

# Unpack results
(best_model_optuna, best_params_optuna, best_score_optuna, study_optuna, 
 X_train_full_optuna, feature_list_full_optuna, importance_df_optuna,
 text_extractor_optuna, best_text_params_optuna) = optuna_results

print("\n" + "="*70)
print("Optimization results unpacked and ready to use!")
print("="*70)

In [ ]:
# Generate predictions on test set using cached features
print("="*70)
print("GENERATING TEST PREDICTIONS (OPTUNA MODEL)")
print("="*70)

# Prepare test text data
text_columns = ['description', 'screen_name', 'location']
test_text = combine_text_columns(test, columns=text_columns)

# Process test data - structured features
print("\nProcessing structured features on test dataset...")
X_test_optuna_struct = process_data(test, verbose=False, expected_features=feature_list_full_optuna)
print(f"  Structured features: {X_test_optuna_struct.shape}")

# Load cached text features for test data (same cache as final model!)
print("Loading text features from cache...")
cache_key = get_cache_key(test_text, best_text_params_optuna)
_, X_test_optuna_text = load_text_features_cache(cache_key)

if X_test_optuna_text is None:
    print("ERROR: Test text features not found in cache!")
    print("This should have been created during final model training.")
    raise RuntimeError("Test text features cache missing")

print(f"  Text features: {X_test_optuna_text.shape}")

# Combine features
print("Combining structured + text features...")
X_test_optuna = np.hstack([X_test_optuna_struct.values, X_test_optuna_text])
print(f"  Combined features: {X_test_optuna.shape}")

# Generate predictions
print("\nGenerating predictions...")
test_probabilities_optuna = best_model_optuna.predict_proba(X_test_optuna)[:, 1]
test_predictions_optuna = (test_probabilities_optuna > 0.5).astype(int)

# Create submission file
submission_optuna = pd.DataFrame({
    'id': test['id'],
    'target': test_predictions_optuna
})

# Save submission
submission_path_optuna = '/Users/winston/Documents/School/Y3S1/BT4012/Kaggle/Competition 1/submissions/CatBoost_NLP_optuna.csv'
submission_optuna.to_csv(submission_path_optuna, index=False)

print(f"\n✓ Predictions generated: {len(test_predictions_optuna)} samples")
print(f"✓ Submission saved to: {submission_path_optuna}")
print(f"✓ Bot predictions: {test_predictions_optuna.sum()}")
print(f"✓ Human predictions: {len(test_predictions_optuna) - test_predictions_optuna.sum()}")

# Save model and study
print("\nSaving trained model and Optuna study...")
joblib.dump(best_model_optuna, '/Users/winston/Documents/School/Y3S1/BT4012/Kaggle/Competition 1/models/trained/catboost_model_nlp_optuna.joblib')
joblib.dump(study_optuna, '/Users/winston/Documents/School/Y3S1/BT4012/Kaggle/Competition 1/models/trained/optuna_study_catboost_nlp.joblib')
joblib.dump(text_extractor_optuna, '/Users/winston/Documents/School/Y3S1/BT4012/Kaggle/Competition 1/models/trained/text_extractor_optuna.joblib')
print("✓ Model, study, and text extractor saved")

In [ ]:
# Visualize Optuna optimization results
# Note: If you get nbformat errors, run: pip install --upgrade nbformat
import joblib
import optuna

study_optuna = joblib.load('trained/optuna_study_catboost_nlp.joblib')
try:
    import plotly
    fig1 = optuna.visualization.plot_optimization_history(study_optuna)
    fig1.show()
    
    fig2 = optuna.visualization.plot_param_importances(study_optuna)
    fig2.show()
except Exception as e:
    print(f"Visualization error: {e}")
    print("\nTo fix this, run in terminal:")
    print("  pip install --upgrade nbformat plotly")
    print("\nAlternatively, you can view the optimization results as text:")
    print(f"\nBest trial: {study_optuna.best_trial.number}")
    print(f"Best value: {study_optuna.best_value:.4f}")
    print(f"Best params: {study_optuna.best_params}")


# Cleanup: Remove NLP Cache Files

Run this cell after completing all model training to free up disk space.

In [ ]:
# Clean up NLP cache files to free disk space
from helpers.nlp_cache import clear_all_caches

print("="*60)
print("CLEANING UP CACHE FILES")
print("="*60)

num_deleted = clear_all_caches()

print(f"\n✓ Cleanup complete! Removed {num_deleted} cache file(s)")
print("\nNote: Cache files speed up model training but can be safely deleted")
print("They will be regenerated automatically when needed.")